# GenAI Evaluation Framework in MLflow 3.14

[MLflow 3.14.0's source code](https://github.com/mlflow/mlflow/tree/v3.14.0)

# Set Up Environment

In [0]:
%pip install -qU "mlflow-skinny[databricks]>=3.14.0" "databricks-agents>=1.11.0" databricks-langchain langchain
%restart_python

In [0]:
import mlflow

displayHTML(f"MLflow version: <b>{mlflow.__version__}</b>")

# mlflow.genai.evaluate


[mlflow.genai.evaluate](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.genai.html#mlflow.genai.evaluate)'s signature ([GitHub](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/evaluation/base.py#L55-L60)):

<br>

```
evaluate(
    data: 'EvaluationDatasetTypes',
    scorers: list[mlflow.genai.scorers.base.Scorer],
    predict_fn: Optional[Callable[..., Any]] = None,
    model_id: str | None = None
) -> 'EvaluationResult'
```

Evaluates the performance of a generative AI model/application using the specified data and scorers:

* `data: 'EvaluationDatasetTypes'` (we'll talk about it later)
* `scorers: list[mlflow.genai.scorers.base.Scorer]` with built-in scorers provided by MLflow and custom scorers (we'll talk about it later, too)
* `predict_fn: Optional[Callable[..., Any]] = None` is for the Gen AI app to be evaluated
* `model_id: str | None = None` is for an AI model

Returns `EvaluationResult` with metrics and detailed per-row assessments (discussed later in this notebook)

Evaluates a model's performance on a given dataset using various scoring criteria.


## Advanced Features on Databricks

As pointed out in the docstring of [mlflow.genai.evaluate](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/evaluation/base.py#L288-L290):

Certain advanced features of this function are only supported on Databricks.

The tracking URI must be `databricks` to use these features.

<!-- https://community.databricks.com/t5/technical-blog/sandbox-magic-turn-your-databricks-notebooks-into-living/ba-p/152352 -->
<!-- https://github.com/stackql-labs/sandbox-magic -->
<div style="border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <span style="font-size: 24px;">⚠️</span>
        <div>
            <strong style="color: #e65100; font-size: 1.1em;">Warning</strong>
            <p style="margin: 8px 0 0 0; color: #333;">I've got no idea what these advanced features are.</p>
        </div>
    </div>
</div>

In [0]:
import mlflow

assert mlflow.get_tracking_uri() == 'databricks'

## mlflow.genai

[mlflow.genai](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.genai.html)

In [0]:
from mlflow.genai import evaluate

help(mlflow.genai.evaluate)


## Code Walkthrough

`evaluate` delegates directly to the internal `_run_harness` for running evaluation.

<br>

<!-- https://community.databricks.com/t5/technical-blog/sandbox-magic-turn-your-databricks-notebooks-into-living/ba-p/152352 -->
<div class="mermaid">
flowchart LR
    A[evaluate] --> B[_run_harness]
    B --> C[harness.run]
    C --> D[Return EvaluationResult]
    style A fill:#E8F4FD,stroke:#5A9BD5,stroke-width:2px
    style D fill:#E8F5E9,stroke:#4CAF50,stroke-width:2px
</div>

<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs";
mermaid.initialize({ startOnLoad: true, theme: "default" });
</script>


<!-- Possibly https://github.com/vbalasu/mermaid-magic -->

# EvaluationDatasetTypes

[EvaluationDatasetTypes](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/evaluation/utils.py#L30-L52) type alias lists the acceptable types of the `data` that `mlflow.genai.evaluate` runs `scorers` on.

<br>

```
evaluate(
    data: 'EvaluationDatasetTypes',
    ...
) -> 'EvaluationResult'
```

<br>

```py
EvaluationDatasetTypes = (
    pd.DataFrame
    | pyspark.sql.dataframe.DataFrame
    | list[dict]
    | list[Trace]
    | mlflow.genai.datasets.EvaluationDataset
    | mlflow.entities.evaluation_dataset.EvaluationDataset
    | mlflow.genai.simulators.ConversationSimulator
    | None
)
```

`EvaluationDatasetTypes` cannot be imported directly since it is guarded by [TYPE_CHECKING](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/evaluation/utils.py#L25-L52) flag.

An interesting part is the [try-except](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/evaluation/utils.py#L30-L52) block.
Unless MLflow runs in a PySpark runtime (e.g., Databricks), `pyspark.sql.dataframe.DataFrame` won't be included among the allowed datasets.

In [0]:
from typing import TYPE_CHECKING

print(TYPE_CHECKING)

## Tale of Two EvaluationDatasets (or Even Three)


There are two `EvaluationDatasetTypes`s with the same name in `EvaluationDatasetTypes` type alias:

* [mlflow.entities.evaluation_dataset.EvaluationDataset](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/entities/evaluation_dataset.py#L38-L631) (alias: `ManagedEvaluationDataset`)
    * First-class entity: create/get/search/delete via MlflowClient/tracking store, supports lazy-loaded records, merge_records, delete_records, granularity validation (trace vs. session)
    * for storing inputs and expectations for GenAI evaluation
    * GenAI evaluation datasets backend entity
    * Persisted records: inputs/outputs/expectations/tags/source, backed by the tracking store
    * Identity: `dataset_id` assigned by the tracking store
    * A backend-facing entity (subclasses `Dataset`, talks to the tracking store directly)
* [mlflow.genai.datasets.EvaluationDataset](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/datasets/evaluation_dataset.py#L17-L285) (alias: `EntityEvaluationDataset`)
    * Public API for dataset evaluation in MLflow's GenAI module.
    * Unified interface for dataset evaluation, supporting both:
        - Standard MLflow evaluation datasets (backed by MLflow's tracking store)
        - Databricks managed datasets (backed by Unity Catalog tables) through the `databricks-agents` library
    * Originally Databricks-only ([Support managed dataset in mlflow.genai.evaluate #15932](https://github.com/mlflow/mlflow/pull/15932)); rewritten in the same [Add Evaluation Datasets to MLflow #17447](https://github.com/mlflow/mlflow/pull/17447) PR to also back onto the new native entity
    * Nothing itself — wraps either the entity class above or a Databricks-managed UC dataset object
    * `to_evaluation_dataset()` converts itself into `mlflow.models.evaluation.EvaluationDataset` so a GenAI dataset can be used in Classic ML evaluation (`mlflow.evaluate()` / `mlflow.models.evaluate()`).

The GenAI evaluation-datasets feature (traces, expectations, curated eval sets stored in the tracking store or Unity Catalog) is a completely separate, much newer subsystem serving `mlflow.genai.evaluate()`.

"the dataset used for evaluation"

Both are subclasses of `Dataset` and `PyFuncConvertibleDatasetMixin`.

[They are aliased at `import`](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/evaluation/utils.py#L26-L27):

<br>

```py
from mlflow.entities.evaluation_dataset import EvaluationDataset as EntityEvaluationDataset
from mlflow.genai.datasets              import EvaluationDataset as ManagedEvaluationDataset
```

Classic ML evaluation (`mlflow.evaluate()` / `mlflow.models.evaluate()`) keeps its old, simple, unpersisted dataset wrapper (`mlflow.models.evaluation.EvaluationDataset`);
the GenAI feature introduces a genuinely new persisted entity plus a public-facing facade over two possible backends.

MLflow chose to keep the intuitive name for all three rather than invent artificial disambiguating names, relying on namespace separation instead.

# EvaluationDataset (mlflow.genai.datasets)

Unified interface for dataset evaluation in MLflow's GenAI module.

<br>

```py
class EvaluationDataset(
    Dataset,
    PyFuncConvertibleDatasetMixin,
)
```

[mlflow/genai/datasets/evaluation_dataset.py](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/datasets/evaluation_dataset.py#L17-L285)

Evaluation Dataset | Internal Property | Description
-|-|-
`mlflow.entities.evaluation_dataset.EvaluationDataset` | `_mlflow_dataset` | Standard MLflow evaluation datasets (backed by MLflow's tracking store)
`databricks.agents.datasets.Dataset` | `_databricks_dataset` | Databricks-managed datasets (backed by Unity Catalog tables)

The `databricks-agents` package is required to use `mlflow.genai.datasets`.

In [0]:
from mlflow.genai.datasets import EvaluationDataset

help(EvaluationDataset)

In [0]:
import mlflow

# traces for evaluation can come from some experiments
# this and few other cells show how to access them

experiments = mlflow.search_experiments()
print(f"Number of experiments: {len(experiments)}")

In [0]:
# it turned out not easy to list experiments
# by just display(experiments)
# this is where Genie Code helped enormously

import pandas as pd

experiments_list = [exp.__dict__ for exp in experiments]
display(pd.DataFrame(experiments_list))

In [0]:
# What's interesting is that the current notebook's experiment can be queried using this method
# See https://github.com/mlflow/mlflow/issues/24387

from mlflow.tracking.fluent import _get_experiment_id

current_notebook_experiment_id = _get_experiment_id()

In [0]:
%sql

CREATE CATALOG IF NOT EXISTS jacek_laskowski;
CREATE SCHEMA IF NOT EXISTS jacek_laskowski.mlflow;

In [0]:
%sql

DROP TABLE IF EXISTS jacek_laskowski.mlflow.qa_dataset

In [0]:
from mlflow.genai.datasets import create_dataset

dataset = create_dataset(
    # In Databricks, the dataset name must be a UC table name in the format: catalog.schema.table
    name="jacek_laskowski.mlflow.qa_dataset",
    experiment_id=current_notebook_experiment_id,
    # Tags are not supported in Databricks environments.
    # Tags are managed through Unity Catalog.
    # tags={
    #     "version": "1.0",
    #     "purpose": "regression_testing",
    #     "model": "gpt-4",
    #     "team": "ml-platform",
    # },
)
print(f"Created dataset: {dataset.dataset_id}")

In [0]:
# Datasets of traces for evaluation can be attached to experiments
# they can be attached to multiple experiments
# via UI or programmatically (using create_dataset)

from mlflow.genai.datasets import search_datasets

# Returns ALL datasets in your tracking server.
# In Databricks, at least one `experiment_id` filter must be provided.
# Listing across all experiments is not supported.

experiment_ids = [current_notebook_experiment_id]
datasets = search_datasets(experiment_ids=experiment_ids)
print(f"Number of datasets: {len(datasets)}")

# Open up PyCharm and review the API
# mlflow.genai.datasets.EvaluationDataset

## mlflow.genai.datasets API


[mlflow/genai/datasets/\_\_init__.py](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/datasets/__init__.py#L768-L778)

<br>

```py
__all__ = [
    "EvaluationDataset",
    "add_dataset_to_experiments",
    "create_dataset",
    "delete_dataset",
    "delete_dataset_tag",
    "get_dataset",
    "remove_dataset_from_experiments",
    "search_datasets",
    "set_dataset_tags",
]
```

In [0]:
from mlflow.genai.datasets import add_dataset_to_experiments

help(add_dataset_to_experiments)

In [0]:
from mlflow.genai.datasets import create_dataset

# In Databricks, databricks.agents.datasets.create_dataset
# Outside Databricks, mlflow.tracking.client.MlflowClient.create_dataset

# NOTE: tags are not supported in Databricks

help(create_dataset)

In [0]:
from mlflow.genai.datasets import get_dataset

# In Databricks, databricks.agents.datasets.get_dataset
# Outside Databricks, mlflow.tracking.client.MlflowClient.get_dataset

help(get_dataset)

In [0]:
from mlflow.genai.datasets import remove_dataset_from_experiments

# mlflow.tracking.client.MlflowClient.remove_dataset_from_experiments

# NOTE: No Databricks-specific remove_dataset_from_experiments since datasets are tables in Unity Catalog

help(remove_dataset_from_experiments)

In [0]:
from mlflow.genai.datasets import search_datasets

# mlflow.tracking.client.MlflowClient.search_datasets

help(search_datasets)

## Input dataset column names

[InputDatasetColumn](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/evaluation/constant.py#L27-L36):
* `request_id`
* `inputs` must be a dictionary of field names and values. For example: `{'query': 'What is MLflow?'}`.
* `request`
* `response`
* `outputs`
* `expectations`
* `tags`
* `trace`
* `source`


# Scorers

[Scorers and LLM Judges in MLflow]($./01_Scorers and LLM Judges in MLflow)

# EvaluationResult

[EvaluationResult](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/evaluation/entities.py#L255-L332) is a Python `@dataclass`.

The evaluation result object with metrics and assessments (available as `metrics` and `result_df` properties, respectively).

In [0]:
print(result)

In [0]:
display(result.result_df)

# Environment Variables

## MLFLOW_GENAI_EVAL_MAX_WORKERS

Maximum number of workers to use for running model prediction and scoring for each row in the dataset passed to the `mlflow.genai.evaluate` function.

In [0]:
from mlflow.environment_variables import MLFLOW_GENAI_EVAL_MAX_WORKERS

print(MLFLOW_GENAI_EVAL_MAX_WORKERS)

## MLFLOW_GENAI_EVAL_MAX_SCORER_WORKERS

Maximum number of concurrent scorer workers to use when running multiple scorers in parallel for each evaluation item.

This helps prevent rate limiting errors when using external LLM APIs as judges.

The actual number of workers will not exceed the number of scorers being used.

When combined with `MLFLOW_GENAI_EVAL_MAX_WORKERS`, the total concurrent scorer invocations is bounded by the product of both values.

Set to 1 to run scorers sequentially.

In [0]:
from mlflow.environment_variables import MLFLOW_GENAI_EVAL_MAX_SCORER_WORKERS

print(MLFLOW_GENAI_EVAL_MAX_SCORER_WORKERS)

## MLFLOW_GENAI_EVAL_ENABLE_SCORER_TRACING

Enable tracing for evaluation scorers.

By default (`False`), MLflow will not trace the scorer function calls.

To trace the scorer functions for debugging purpose, set this to `True`.

In [0]:
from mlflow.environment_variables import MLFLOW_GENAI_EVAL_ENABLE_SCORER_TRACING

print(MLFLOW_GENAI_EVAL_ENABLE_SCORER_TRACING)

# How to Use

There are ~~three~~ four different ways to use this function (see [mlflow.genai.evaluate](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.genai.html#mlflow.genai.evaluate)):

1. Use Traces to evaluate the model/application.
1. Use DataFrame or dictionary with “inputs”, “outputs”, “expectations” columns.
1. Pass `predict_fn` and input samples (and optionally expectations).
1. Pass a ConversationSimulator for multi-turn evaluation.

## Traces

**MLflow Trace Objects** contain the execution to evaluate.

When provided, scorers will use them to extract necessary data to evaluate

## ConversationSimulator

⚠️ Not covered in this notebook

<br>

```py
from mlflow.genai.simulators import ConversationSimulator
```

Available as `mlflow.genai.ConversationSimulator` (through [mlflow/genai/\_\_init__.py](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/__init__.py#L95))

Special support of `mlflow.genai.evaluate` for `ConversationSimulator`:
* `predict_fn` is required when using `ConversationSimulator` as data (in `mlflow.genai.evaluate`) for the simulator to generate conversations.

### Purpose by Claude

Generates multi-turn synthetic conversations between an LLM-simulated "user" and your agent's `predict_fn`, tracing every turn in MLflow.

Designed to plug directly into `mlflow.genai.evaluate(data=simulator, ...)` as a data source, so you can run scorers (e.g. `Safety`, `ConversationalSafety`) over realistic multi-turn dialogues instead of hand-writing them.

---

prompt: "brief me about ConversationSimulator in mlflow.genai.simulators"

In [0]:
from mlflow.genai import ConversationSimulator

help(ConversationSimulator)

## Example: Automatic Trace Evaluation

There are a few scorers registered earlier in this notebook:

1. `RelevanceToQuery`
1. `output_length_checker`

Go to **Judges** in the MLflow UI for this notebook experiment.

You're now going to autolog an LLM application.
This will be a [simple LangChain agent with a custom tool](https://docs.langchain.com/oss/python/langchain/overview#create-an-agent).

In [0]:
%pip show deepagents

In [0]:
import mlflow

mlflow.langchain.autolog()

In [0]:
%pip show databricks-langchain

In [0]:
from databricks_langchain import ChatDatabricks

LLM_ENDPOINT_NAME = "databricks-claude-opus-4-8"

databricks_chat_model = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

In [0]:
help(registered_scorer.start)

In [0]:
# Start scorer with 100% sampling rate

from mlflow.genai.scorers import ScorerSamplingConfig

active_scorer = registered_scorer.start(sampling_config=ScorerSamplingConfig(sample_rate=1))
print(f"Scorer is evaluating {active_scorer.sample_rate * 100}% of traces")

In [0]:
# Create a simple LangChain agent with a custom tool
# https://docs.langchain.com/oss/python/langchain/overview#create-an-agent

from langchain.agents import create_agent

def check_weather(city: str) -> str:
    """Get weather forecast for the given city."""
    if city == "Paris, France":
        return "cloudy"
    elif city == "Rome, Italy":
        return "rainy"
    else:
        return f"It's always sunny in {city}!"

langchain_weather_agent = create_agent(
    # model: str | BaseChatModel,
    model=databricks_chat_model,
    # tools: Sequence[BaseTool | Callable[..., Any] | dict[str, Any]] | None = None,
    tools=[check_weather],
    # system_prompt: str | SystemMessage | None = None,
    system_prompt="You are a weather assistant. Be concise (less than 5 words) and speak English only.",
)

city = "San Francisco"
result = langchain_weather_agent.invoke(
    {"messages": [{"role": "user", "content": f"What's the weather in {city}?"}]}
)
print(result["messages"][-1].content_blocks)

In [0]:
import mlflow

from mlflow.genai import scorer
from mlflow.genai.scorers import Correctness, Guidelines


# Define a dataset for evaluation
eval_dataset = [
    {
        "inputs": {
            "city": "Paris, France"
        },
        "expectations": {
            "expected_response": "cloudy"
        },
    },
    {
        "inputs": {
            "city": "Warsaw, Poland"
        },
        "expectations": {
            "expected_response": "sunny"
        },
    },
    {
        "inputs": {
            "city": "Rome, Italy"
        },
        "expectations": {
            "expected_response": "rainy"
        },
    },
]

# Define a custom scorer
@scorer
def is_concise(outputs: str) -> bool:
    """Evaluate if the answer is concise (less than 5 words)"""
    return len(outputs) <= 5

# Define evaluation criteria (scorers)
scorers = [
    Correctness(),
    Guidelines(name="is_english", guidelines="The answer must be in English"),
    is_concise,
]

# Define a prediction function with your agent
def complete_agentic_workflow(city: str) -> str:
    result = langchain_weather_agent.invoke(
        {"messages": [{"role": "user", "content": f"What's the weather in {city}?"}]}
    )
    return result["messages"][-1].content_blocks

# All three components of the evaluation: dataset, prediction function, and scorers defined.
# Let's run the evaluation!
evaluation_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=complete_agentic_workflow,
    scorers=scorers,
)

In [0]:
print(evaluation_results)

In [0]:
display(evaluation_results.result_df.drop("assessments", axis=1))

# Debugging Evaluation Harness

Execute the following cell and re-run evaluation (**Run Evaluation** cell).

In [0]:
from mlflow.genai.evaluation import harness

harness._logger.setLevel("DEBUG")